# 🥁 HT-Demucs FT: Drums Specialist Playground

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shinohara-rin/stuff/blob/main/htdemucs_drums_playground.ipynb)

A hands-on playground for [`StemSplitio/htdemucs-ft-drums-pytorch`](https://huggingface.co/StemSplitio/htdemucs-ft-drums-pytorch):
**sub-model 0 of Meta's 4-model `htdemucs_ft` bag**, the one trained to pull out **drums**.
It's about a quarter of the full ensemble's size and compute, and its drums output is the same (median SDR ≈ 10.1 dB on MUSDB18-HQ).

**What you can do here**
1. Load the drums specialist (on a GPU if one is available)
2. Feed it audio: a built-in demo track, your own upload, a direct URL, or YouTube
3. Separate it into **drums** and **everything else**, and listen to both
4. Look at waveforms and spectrograms
5. Remix the drum level up or down
6. Analyse the drum stem: tempo, beats, onsets, a click track, and auto-sliced one-shots
7. **🔥 Auto-breakcore remixer**: chop, stutter, reverse and re-sequence the drums at 180+ BPM
8. Download everything as a zip
9. **🎛️ Gradio app** behind a fast Cloudflare tunnel: separate, remix and breakcore from a web UI
10. *(Optional)* Compare against the full 4-model ensemble

> 💡 **Tip:** In Colab, go to *Runtime → Change runtime type → T4 GPU*. A 3-minute song takes seconds on a GPU and around a minute on CPU.

---

## 1 · Setup

In [ ]:
# Demucs 4.0.1 is the version the model card / handler.py targets.
!pip install -q demucs==4.0.1 yt-dlp

In [ ]:
import os, io, time, shutil, subprocess, zipfile, urllib.request
import numpy as np
import torch
import soundfile as sf
import librosa, librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown

from demucs.pretrained import get_model
from demucs.apply import apply_model
from demucs.audio import convert_audio

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
print(f"torch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""))
if device == "cpu":
    print("⚠️  No GPU found. This still works, just slower. Runtime → Change runtime type → GPU.")

WORK = "/content" if IN_COLAB else os.path.abspath(".")
IN_DIR, OUT_DIR = os.path.join(WORK, "inputs"), os.path.join(WORK, "outputs")
os.makedirs(IN_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

# In-notebook players embed MP3 rather than WAV (~10x smaller) so the notebook stays light.
# Saved files in section 10 are still WAV.
import lameenc
PREVIEW_KBPS = 192

def to_mp3(wav, rate, kbps=PREVIEW_KBPS):
    x = wav.numpy() if torch.is_tensor(wav) else np.asarray(wav)
    x = np.atleast_2d(x).astype(np.float32)[:2]
    peak = np.abs(x).max()
    if peak > 1:
        x = x / peak
    enc = lameenc.Encoder()
    enc.set_bit_rate(kbps); enc.set_in_sample_rate(int(rate))
    enc.set_channels(x.shape[0]); enc.set_quality(2)
    pcm = (np.clip(x, -1, 1).T * 32767).astype("<i2").tobytes()
    return bytes(enc.encode(pcm) + enc.flush())

def player(label, wav, rate=None):
    if label:
        display(Markdown(f"**{label}**"))
    a = Audio(data=to_mp3(wav, rate or SR))
    a.mimetype = "audio/mpeg"
    display(a)

## 2 · Load the drums specialist

The HF repo doesn't ship separate weights. Its `handler.py` loads the official `htdemucs_ft` bag through Demucs and keeps only `bag.models[0]`. We do the same here. The first run downloads about 320 MB of checkpoints from Meta's CDN.

In [ ]:
import hashlib, yaml
from demucs.pretrained import REMOTE_ROOT, _parse_remote_files

def prefetch_checkpoints(bag_name="htdemucs_ft"):
    # Download the bag's checkpoints into the torch.hub cache ourselves, verifying the
    # sha256 prefix in the filename and retrying on a bad download (torch.hub would
    # just raise "invalid hash value"). Demucs then loads them from the cache.
    urls = _parse_remote_files(REMOTE_ROOT / "files.txt")
    with open(REMOTE_ROOT / f"{bag_name}.yaml") as f:
        sigs = yaml.safe_load(f)["models"]
    ckpt_dir = os.path.join(torch.hub.get_dir(), "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    for sig in sigs:
        url = urls[sig]
        fname = url.rsplit("/", 1)[1]                  # e.g. f7e0c4bc-ba3fe64a.th
        dst = os.path.join(ckpt_dir, fname)
        if os.path.exists(dst):
            continue
        tmp = dst + ".partial"
        expected = fname.rsplit(".", 1)[0].split("-")[-1]
        for attempt in range(3):
            torch.hub.download_url_to_file(url, tmp, progress=True)
            with open(tmp, "rb") as f:
                digest = hashlib.sha256(f.read()).hexdigest()
            if digest.startswith(expected):
                os.replace(tmp, dst)
                break
            print(f"Checksum mismatch for {fname} (attempt {attempt+1}), retrying…")
        else:
            os.remove(tmp)
            raise RuntimeError(f"Could not download a valid {fname}")

t0 = time.time()
prefetch_checkpoints("htdemucs_ft")
bag = get_model("htdemucs_ft")         # 4-model bag; each sub-model specialises in one stem
model = bag.models[0].eval().to(device)  # sub-model 0 = drums specialist

SR       = int(bag.samplerate)           # 44100
CHANNELS = int(bag.audio_channels)       # 2
SOURCES  = list(bag.sources)             # ['drums', 'bass', 'other', 'vocals']
DRUMS    = SOURCES.index("drums")

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Params: {n_params/1e6:.1f} M  (~{n_params*4/2**20:.0f} MB fp32)")
print(f"Sample rate: {SR} Hz | channels: {CHANNELS} | sources: {SOURCES}")
print(f"Bag weights for sub-model 0: {dict(zip(SOURCES, bag.weights[0]))}")
print(f"Native segment length: {float(model.segment):.2f}s")

## 3 · Pick your audio

Choose a `SOURCE`:
- **demo**: royalty-free tracks from the [librosa example set](https://librosa.org/doc/latest/recordings.html) (CC-BY / CC0)
- **upload**: a file from your computer (mp3, wav, flac, ogg, m4a, …)
- **url**: a direct link to an audio file
- **youtube**: any URL `yt-dlp` can handle. Only use audio you have the rights to.

`START_S` / `DURATION_S` trim the input so experiments stay quick. Set `DURATION_S = 0` to process the whole file.

In [ ]:
SOURCE     = "demo"     # @param ["demo", "upload", "url", "youtube"]
DEMO_TRACK = "vibeace"  # @param ["vibeace", "fishin", "choice", "sweetwaltz"]
URL        = ""         # @param {type:"string"}
START_S    = 0          # @param {type:"number"}
DURATION_S = 30         # @param {type:"number"}


def fetch_input(source, url="", demo="vibeace"):
    if source == "demo":
        return librosa.ex(demo)
    if source == "upload":
        if not IN_COLAB:
            raise RuntimeError("Upload widget is Colab-only; put a file next to the notebook and use SOURCE='url' with a file:// path.")
        up = colab_files.upload()
        name = next(iter(up))
        path = os.path.join(IN_DIR, os.path.basename(name))
        with open(path, "wb") as f:
            f.write(up[name])
        return path
    if not url:
        raise ValueError("Set URL for SOURCE='url' / 'youtube'.")
    if source == "url":
        path = os.path.join(IN_DIR, os.path.basename(url.split("?")[0]) or "download")
        urllib.request.urlretrieve(url, path)
        return path
    if source == "youtube":
        import yt_dlp
        opts = {"format": "bestaudio/best", "outtmpl": os.path.join(IN_DIR, "%(title).80s [%(id)s].%(ext)s"),
                "quiet": True, "noplaylist": True}
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=True)
            print("Downloaded:", info.get("title"))
            return ydl.prepare_filename(info)
    raise ValueError(source)


def load_audio(path, start=0.0, duration=0.0):
    # Decode with ffmpeg (handles almost any format), trim, and resample to the model rate.
    cmd = ["ffmpeg", "-v", "error", "-ss", str(start), "-i", path]
    if duration and duration > 0:
        cmd += ["-t", str(duration)]
    cmd += ["-f", "f32le", "-acodec", "pcm_f32le", "-ac", str(CHANNELS), "-ar", str(SR), "-"]
    raw = subprocess.run(cmd, check=True, capture_output=True).stdout
    wav = np.frombuffer(raw, dtype=np.float32).reshape(-1, CHANNELS).T.copy()
    return torch.from_numpy(wav)  # (channels, samples)


in_path = fetch_input(SOURCE, URL, DEMO_TRACK)
mix = load_audio(in_path, START_S, DURATION_S)
if mix.shape[-1] == 0:
    raise ValueError("Got no audio. Check START_S / DURATION_S against the track length.")
TRACK_NAME = os.path.splitext(os.path.basename(in_path))[0].replace(" ", "_")
print(f"{TRACK_NAME}: {mix.shape[-1]/SR:.1f}s, {mix.shape[0]} ch @ {SR} Hz")
player(None, mix)

## 4 · Separate

Knobs (all standard `demucs.apply.apply_model` options):
- **`SHIFTS`**: random time-shift augmentation, averaged. Each shift adds one more full pass. Usually worth +0.1–0.3 dB SDR; `0` is fastest.
- **`OVERLAP`**: overlap between chunks. Higher means smoother joins and slower runs.
- **`SPLIT`**: process in chunks of the model's native segment length (keep this on for long tracks or low VRAM).

In [ ]:
SHIFTS  = 1     # @param {type:"slider", min:0, max:5, step:1}
OVERLAP = 0.25  # @param {type:"slider", min:0.0, max:0.75, step:0.05}
SPLIT   = True  # @param {type:"boolean"}

def run_model(m, mix, shifts=1, split=True, overlap=0.25, progress=True):
    # Demucs normalises the input itself during training; matching that at inference
    # (as demucs.separate does) helps on quiet/loud tracks.
    ref = mix.mean(0)
    mean, std = ref.mean(), ref.std() + 1e-8
    with torch.inference_mode():
        stems = apply_model(m, ((mix - mean) / std).unsqueeze(0), device=device,
                            shifts=shifts, split=split, overlap=overlap, progress=progress)[0]
    return (stems * std + mean).cpu()


def separate(mix, shifts=1, split=True, overlap=0.25, progress=True):
    # Returns (drums, no_drums). Rows other than DRUMS in the model output are weak
    # by-products (the bag gives them weight 0), so "everything else" is the residual.
    drums = run_model(model, mix, shifts, split, overlap, progress)[DRUMS]
    return drums, mix - drums


if device == "cuda":
    torch.cuda.synchronize()
t0 = time.time()
drums, no_drums = separate(mix, SHIFTS, SPLIT, OVERLAP)
if device == "cuda":
    torch.cuda.synchronize()
elapsed = time.time() - t0

dur = mix.shape[-1] / SR
print(f"Separated {dur:.1f}s of audio in {elapsed:.1f}s → RTF {elapsed/dur:.3f} "
      f"({dur/elapsed:.1f}× real-time) on {device}")

## 5 · Listen

In [ ]:
player("Original mix", mix)
player("🥁 Drums (isolated)", drums)
player("🎸 Everything except drums (mix − drums)", no_drums)

## 6 · Visualise

In [ ]:
VIEW_START_S = 0   # @param {type:"number"}
VIEW_LEN_S   = 10  # @param {type:"number"}

a, b = int(VIEW_START_S * SR), int((VIEW_START_S + VIEW_LEN_S) * SR)
sigs = {"Mix": mix, "Drums": drums, "No drums": no_drums}
fig, axes = plt.subplots(2, 3, figsize=(18, 7), sharex="col")
for i, (name, w) in enumerate(sigs.items()):
    seg = w.mean(0)[a:b].numpy()
    t = np.arange(len(seg)) / SR + VIEW_START_S
    axes[0, i].plot(t, seg, lw=0.4)
    axes[0, i].set_ylim(-1, 1); axes[0, i].set_title(name)
    S = librosa.amplitude_to_db(np.abs(librosa.stft(seg, n_fft=2048, hop_length=512)), ref=np.max)
    librosa.display.specshow(S, sr=SR, hop_length=512, x_axis="time", y_axis="log",
                             ax=axes[1, i], vmin=-80, vmax=0)
    axes[1, i].set_xlabel("time (s, from view start)")
axes[0, 0].set_ylabel("amplitude")
plt.tight_layout(); plt.show()

def rms_db(w):
    return 20 * np.log10(w.pow(2).mean().sqrt().item() + 1e-12)
print(f"RMS  mix {rms_db(mix):.1f} dBFS | drums {rms_db(drums):.1f} dBFS | no-drums {rms_db(no_drums):.1f} dBFS")

## 7 · Remix playground

Turn the drums up, down, or off. `-60` effectively mutes them; `+6` gives a drum-forward mix. Re-run the cell after you move the sliders.

In [ ]:
DRUM_GAIN_DB  = 6   # @param {type:"slider", min:-60, max:12, step:1}
OTHER_GAIN_DB = 0   # @param {type:"slider", min:-60, max:12, step:1}

g_d, g_o = 10 ** (DRUM_GAIN_DB / 20), 10 ** (OTHER_GAIN_DB / 20)
remix = g_d * drums + g_o * no_drums
peak = remix.abs().max().item()
if peak > 1:
    remix = remix / peak * 0.99
    print(f"(peak {peak:.2f} > 1, normalised to avoid clipping)")
player(f"Remix: drums {DRUM_GAIN_DB:+d} dB, rest {OTHER_GAIN_DB:+d} dB", remix)

## 8 · Drum analysis: tempo, beats, onsets, one-shots

One of the model's main uses is feeding a clean drum stem into beat and onset detection. Here we compare beat tracking on the **mix** and on the **isolated drums**, render a click track over the drums, and auto-slice the strongest hits into one-shot samples.

In [ ]:
N_ONESHOTS   = 8    # @param {type:"slider", min:1, max:32, step:1}
ONESHOT_MS   = 250  # @param {type:"slider", min:50, max:1000, step:10}

ASR = 22050  # analysis rate
d_mono = librosa.resample(drums.mean(0).numpy(), orig_sr=SR, target_sr=ASR)
m_mono = librosa.resample(mix.mean(0).numpy(),   orig_sr=SR, target_sr=ASR)

for name, y in [("mix", m_mono), ("drums", d_mono)]:
    tempo, _ = librosa.beat.beat_track(y=y, sr=ASR)
    print(f"Tempo estimate from {name:5s}: {float(np.atleast_1d(tempo)[0]):6.1f} BPM")

env = librosa.onset.onset_strength(y=d_mono, sr=ASR)
tempo, beats = librosa.beat.beat_track(onset_envelope=env, sr=ASR)
onsets = librosa.onset.onset_detect(onset_envelope=env, sr=ASR, backtrack=True)
beat_t, onset_t = librosa.frames_to_time(beats, sr=ASR), librosa.frames_to_time(onsets, sr=ASR)
print(f"{len(beat_t)} beats, {len(onset_t)} drum onsets detected")

fig, ax = plt.subplots(figsize=(18, 3))
t_env = librosa.times_like(env, sr=ASR)
ax.plot(t_env, env / env.max(), lw=0.8, label="onset strength (drums)")
ax.vlines(beat_t, 0, 1, color="r", alpha=0.5, lw=1, label="beats")
ax.vlines(onset_t, 0, 0.3, color="k", alpha=0.6, lw=0.8, label="onsets")
ax.set_xlim(VIEW_START_S, VIEW_START_S + VIEW_LEN_S); ax.set_xlabel("time (s)"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

clicks = librosa.clicks(times=beat_t, sr=ASR, length=len(d_mono), click_freq=1500)
player("Drums + beat click track", np.stack([d_mono, d_mono]) * 0.8 + clicks * 0.3, rate=ASR)

# One-shots: take the strongest onsets, keeping each slice from overlapping the next one.
slice_len = int(ONESHOT_MS / 1000 * SR)
d_full = drums.numpy()
onset_samples = (onset_t * SR).astype(int)
strength = env[onsets] if len(onsets) else np.array([])
picked, taken = [], np.zeros(d_full.shape[-1], bool)
for idx in np.argsort(-strength):
    s = onset_samples[idx]
    if s + slice_len <= d_full.shape[-1] and not taken[s:s + slice_len].any():
        picked.append(s); taken[s:s + slice_len] = True
    if len(picked) >= N_ONESHOTS:
        break

fade = np.ones(slice_len, np.float32)
fl = min(slice_len // 4, int(0.02 * SR)); fade[-fl:] = np.linspace(1, 0, fl)
ONESHOTS = []
for s in sorted(picked):
    hit = d_full[:, s:s + slice_len] * fade
    ONESHOTS.append((s / SR, hit / (np.abs(hit).max() + 1e-9) * 0.9))

display(Markdown(f"**{len(ONESHOTS)} auto-sliced one-shots** (peak-normalised)"))
if ONESHOTS:
    gap = np.zeros((CHANNELS, int(0.15 * SR)), np.float32)
    player(None, np.concatenate(sum([[h, gap] for _, h in ONESHOTS], []), axis=1))
    print("at:", ", ".join(f"{t:.2f}s" for t, _ in ONESHOTS))

## 9 · 🔥 Auto-breakcore remixer

Now we put the beat grid to work. The drum stem is cut into **16th-note slices** along the detected beats. They're re-sequenced at a much faster tempo, then a seeded random "chaos engine" mangles them with **stutters, reverses, jumps, pitch-shifted slices and gaps**, plus a **snare-roll fill** at the end of every phrase. The no-drums stem is re-gridded underneath, so the harmony follows the chopped drums bar by bar.

| Knob | What it does |
|---|---|
| `TARGET_BPM` | Output tempo. Slices are sped up (repitched) to fit, jungle-style. |
| `CHAOS` | 0 = a straight re-tempo, 1 = full mayhem |
| `SEED` | Change it to re-roll the pattern. The same seed always gives the same track. |
| `BED` | The music under the drums: `sped-up` (nightcore pitch), `original pitch` (pitch-corrected) or `off` |
| `KICK` | Layers a synthesised distorted kick: `jungle` (1 … &3-ish), `gabber` (four-on-the-floor) or `off` |
| `DRIVE` / `CRUSH_BITS` / `CRUSH_HOLD` | Drum-bus saturation, bit depth, and sample-rate reduction |

Re-run the cell after you change anything.

In [ ]:
# Breakcore engine. Defined as a function so the Gradio app in section 11 can reuse it.
ACTIONS = ["play", "stutter", "reverse", "jump", "pitch", "gate", "fill"]
HOP = 512


def _read(src, start, n_read, n_out):
    # Read n_read samples from src at start and resample them to n_out (repitch).
    n_read = max(int(n_read), 2)
    seg = src[:, start:start + n_read]
    if seg.shape[1] < n_read:
        seg = np.pad(seg, ((0, 0), (0, n_read - seg.shape[1])))
    xi = np.linspace(0, n_read - 1, n_out)
    return np.stack([np.interp(xi, np.arange(n_read), c) for c in seg]).astype(np.float32)


def _declick(x, ms=2):
    f = min(int(SR * ms / 1000), x.shape[1] // 2)
    if f > 0:
        ramp = np.linspace(0, 1, f, dtype=np.float32)
        x[:, :f] *= ramp; x[:, -f:] *= ramp[::-1]
    return x


def breakcore_remix(drums, no_drums, target_bpm=180, out_bars=16, chaos=0.6, seed=7,
                    fill_every=4, bed="sped-up", bed_gain_db=-4, kick="jungle",
                    drive=2.5, crush_bits=12, crush_hold=1):
    # Returns (audio (ch, n) tensor, grid (bars, 16) of ACTIONS ids, source bar per output bar, info).
    rng = np.random.default_rng(int(seed))
    d_src, bed_src = drums.numpy(), no_drums.numpy()

    # 1. Beat grid on the isolated drums, split into 16th-note slices
    env = librosa.onset.onset_strength(y=d_src.mean(0), sr=SR, hop_length=HOP)
    src_tempo, beat_frames = librosa.beat.beat_track(onset_envelope=env, sr=SR, hop_length=HOP)
    src_tempo = float(np.atleast_1d(src_tempo)[0]) or 120.0
    beat_pos = librosa.frames_to_samples(beat_frames, hop_length=HOP)
    if len(beat_pos) < 9:  # beat tracker found too little: fall back to a straight grid
        period = int(SR * 60 / src_tempo)
        beat_pos = np.arange(0, d_src.shape[-1] - period, period)

    starts, lens = [], []
    for a, b in zip(beat_pos[:-1], beat_pos[1:]):
        q = (b - a) / 4
        starts += [int(a + j * q) for j in range(4)]
        lens += [int(q)] * 4
    n_src_bars = len(starts) // 16
    if n_src_bars < 1:
        raise ValueError("Need at least one full bar (4 beats) of audio. Use a longer clip.")
    K = n_src_bars * 16
    starts, lens = np.array(starts[:K]), np.array(lens[:K])
    slice_strength = env[np.minimum(starts // HOP, len(env) - 1)]

    L = int(round(SR * 60 / target_bpm / 4))  # output samples per 16th

    def roll(k, n, semis=0.0):
        # Repeat the head of slice k n times inside one step (stutter / snare roll).
        piece = _declick(_read(d_src, starts[k], lens[k] / n * 2 ** (semis / 12), L // n))
        out = np.concatenate([piece * g for g in np.linspace(0.55, 1.0, n, dtype=np.float32)], axis=1)
        return np.pad(out, ((0, 0), (0, L - out.shape[1])))

    # 2. Chaos engine: one action per 16th step
    base_p = np.array([0.18, 0.12, 0.25, 0.12, 0.06]) * chaos  # stutter, reverse, jump, pitch, gate
    p = np.concatenate([[1 - base_p.sum()], base_p])

    drum_out = np.zeros((CHANNELS, out_bars * 16 * L), np.float32)
    grid = np.zeros((out_bars, 16), int)
    bar_src = []
    for r in range(out_bars):
        sb = r % n_src_bars
        if rng.random() < 0.3 * chaos:
            sb = int(rng.integers(n_src_bars))  # bar shuffle
        bar_src.append(sb)
        fill_bar = fill_every and (r + 1) % fill_every == 0
        k_snare = sb * 16 + int(np.argmax(slice_strength[sb * 16 + 4: sb * 16 + 16])) + 4
        for i in range(16):
            k = sb * 16 + i
            act = "fill" if fill_bar and i >= 12 else str(rng.choice(ACTIONS[:6], p=p))
            if act == "play":
                seg = _read(d_src, starts[k], lens[k], L)
            elif act == "jump":
                k = int(rng.integers(K)); seg = _read(d_src, starts[k], lens[k], L)
            elif act == "reverse":
                seg = _read(d_src, starts[k], lens[k], L)[:, ::-1].copy()
            elif act == "pitch":
                semis = rng.choice([-12, -7, -5, 5, 7, 12])
                seg = _read(d_src, starts[k], lens[k] * 2 ** (semis / 12), L)
            elif act == "stutter":
                seg = roll(k, int(rng.choice([2, 3, 4, 8])))
            elif act == "gate":
                seg = np.zeros((CHANNELS, L), np.float32)
            else:  # fill: snare roll that gets denser and climbs in pitch
                j = i - 12
                seg = roll(k_snare, [4, 6, 8, 16][j], semis=[0, 2, 4, 7][j])
            grid[r, i] = ACTIONS.index(act)
            o = (r * 16 + i) * L
            drum_out[:, o:o + L] = _declick(seg)

    # 3. Synth kick layer
    kick_steps = {"jungle": [0, 10], "gabber": [0, 4, 8, 12], "off": []}[kick]
    if kick_steps:
        t = np.arange(int(0.25 * SR)) / SR
        freq = 45 + 110 * np.exp(-t * 35)
        kick_wav = np.sin(2 * np.pi * np.cumsum(freq) / SR) * np.exp(-t * (9 if kick == "gabber" else 14))
        kick_wav = np.tanh(kick_wav * (6 if kick == "gabber" else 2)).astype(np.float32)
        for r in range(out_bars):
            for i in kick_steps:
                if grid[r, i] in (ACTIONS.index("gate"), ACTIONS.index("fill")):
                    continue
                o = (r * 16 + i) * L
                n = min(len(kick_wav), drum_out.shape[1] - o)
                drum_out[:, o:o + n] += 0.6 * kick_wav[:n]

    # 4. Drum-bus FX: drive, bitcrush, sample-rate reduction
    drum_out /= np.abs(drum_out).max() + 1e-9
    drum_out = np.tanh(drum_out * drive) / np.tanh(drive)
    if crush_bits < 16:
        q = 2 ** (crush_bits - 1)
        drum_out = np.round(drum_out * q) / q
    if crush_hold > 1:
        drum_out = np.repeat(drum_out[:, ::crush_hold], crush_hold, axis=1)[:, :drum_out.shape[1]]

    # 5. Music bed: the no-drums stem, re-gridded to follow the drum source bars
    speed = np.median(np.diff(beat_pos)) / (4 * L)
    bed_out = np.zeros_like(drum_out)
    if bed != "off":
        for r, sb in enumerate(bar_src):
            for bt in range(4):
                b = sb * 4 + bt
                a0, a1 = beat_pos[b], beat_pos[b + 1]
                if rng.random() < 0.15 * chaos or (fill_every and (r + 1) % fill_every == 0 and bt == 3):
                    piece = _declick(_read(bed_src, a0, (a1 - a0) / 4, L))  # glitch: loop the beat's head
                    seg = np.concatenate([piece] * 4, axis=1)
                else:
                    seg = _read(bed_src, a0, a1 - a0, 4 * L)
                o = (r * 4 + bt) * 4 * L
                bed_out[:, o:o + 4 * L] = _declick(seg, ms=4)
        if bed == "original pitch":
            bed_out = librosa.effects.pitch_shift(bed_out, sr=SR, n_steps=-12 * np.log2(speed))
        bed_out *= 0.9 / (np.abs(bed_out).max() + 1e-9)

    # 6. Master: sum, soft-clip, normalise
    out = drum_out + 10 ** (bed_gain_db / 20) * bed_out
    out /= np.abs(out).max() + 1e-9
    out = np.tanh(out * 1.5) / np.tanh(1.5)
    audio = torch.from_numpy((out * 0.95 / (np.abs(out).max() + 1e-9)).astype(np.float32))

    counts = {a: int((grid == n).sum()) for n, a in enumerate(ACTIONS)}
    info = (f"Source ≈ {src_tempo:.1f} BPM, {n_src_bars} bars → {target_bpm} BPM "
            f"(slices sped up ×{speed:.2f}, {12*np.log2(speed):+.1f} semitones), "
            f"{out_bars} bars = {audio.shape[-1]/SR:.1f}s\n"
            "Actions: " + ", ".join(f"{a} {c}" for a, c in counts.items() if c))
    return audio, grid, bar_src, info

In [ ]:
TARGET_BPM  = 180       # @param {type:"slider", min:140, max:260, step:1}
OUT_BARS    = 16        # @param {type:"slider", min:4, max:64, step:1}
CHAOS       = 0.6       # @param {type:"slider", min:0.0, max:1.0, step:0.05}
SEED        = 7         # @param {type:"integer"}
FILL_EVERY  = 4         # @param {type:"slider", min:0, max:8, step:1}
BED         = "sped-up" # @param ["sped-up", "original pitch", "off"]
BED_GAIN_DB = -4        # @param {type:"slider", min:-24, max:6, step:1}
KICK        = "jungle"  # @param ["jungle", "gabber", "off"]
DRIVE       = 2.5       # @param {type:"slider", min:1.0, max:10.0, step:0.5}
CRUSH_BITS  = 12        # @param {type:"slider", min:4, max:16, step:1}
CRUSH_HOLD  = 1         # @param {type:"slider", min:1, max:8, step:1}

breakcore, grid, bar_src, bc_info = breakcore_remix(
    drums, no_drums, TARGET_BPM, OUT_BARS, CHAOS, SEED, FILL_EVERY, BED, BED_GAIN_DB,
    KICK, DRIVE, CRUSH_BITS, CRUSH_HOLD)
print(bc_info)
player(f"🔥 Auto-breakcore: seed {SEED}, chaos {CHAOS}", breakcore)

In [ ]:
# Pattern map: one row per output bar, one cell per 16th, coloured and lettered by action.
from matplotlib.colors import ListedColormap
from matplotlib.figure import Figure
from matplotlib.patches import Patch

ACTION_STYLE = {  # colour, letter
    "play":    ("#d6d5cf", ""),
    "stutter": ("#2a78d6", "S"),
    "reverse": ("#eb6834", "R"),
    "jump":    ("#1baf7a", "J"),
    "pitch":   ("#eda100", "P"),
    "gate":    ("#fcfcfb", "·"),
    "fill":    ("#e87ba4", "F"),
}


def plot_pattern(grid, bar_src, title):
    # Built on a bare Figure (not pyplot) so repeated calls from the app don't leak figures.
    n_bars = grid.shape[0]
    fig = Figure(figsize=(14, 0.32 * n_bars + 1.4), facecolor="#fcfcfb")
    ax = fig.subplots()
    ax.set_facecolor("#fcfcfb")
    ax.imshow(grid, cmap=ListedColormap([ACTION_STYLE[a][0] for a in ACTIONS]),
              vmin=0, vmax=len(ACTIONS) - 1, aspect="auto")
    for r in range(n_bars):
        for i in range(16):
            ax.text(i, r, ACTION_STYLE[ACTIONS[grid[r, i]]][1], ha="center", va="center",
                    fontsize=8, color="#0b0b0a")
    ax.set_xticks(np.arange(-0.5, 16, 1), minor=True); ax.set_yticks(np.arange(-0.5, n_bars, 1), minor=True)
    ax.grid(which="minor", color="#fcfcfb", linewidth=2)
    ax.tick_params(which="minor", length=0)
    ax.set_xticks([1.5, 5.5, 9.5, 13.5], [f"beat {n+1}" for n in range(4)])
    ax.set_yticks(range(n_bars), [f"bar {r+1} ← src {s+1}" for r, s in enumerate(bar_src)], fontsize=8)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(colors="#55554f", length=0)
    ax.set_title(title, loc="left", color="#0b0b0a")
    ax.legend(handles=[Patch(facecolor=c, edgecolor="#b5b4ad", label=f"{l} {a}".strip())
                       for a, (c, l) in ACTION_STYLE.items()],
              ncol=7, loc="upper left", bbox_to_anchor=(0, -0.02 - 0.6 / max(n_bars, 1)),
              frameon=False, fontsize=9, labelcolor="#55554f")
    fig.tight_layout()
    return fig


display(plot_pattern(grid, bar_src, f"Chop pattern (seed {SEED}, chaos {CHAOS})"))

## 10 · Save & download

In [ ]:
SUBTYPE = "PCM_16"  # @param ["PCM_16", "PCM_24", "FLOAT"]

def save(name, wav, rate=SR):
    arr = wav.numpy() if torch.is_tensor(wav) else wav
    if SUBTYPE != "FLOAT":
        arr = np.clip(arr, -1, 1)
    path = os.path.join(OUT_DIR, name)
    sf.write(path, arr.T, rate, subtype=SUBTYPE)
    return path

paths = [save(f"{TRACK_NAME}_drums.wav", drums),
         save(f"{TRACK_NAME}_no_drums.wav", no_drums),
         save(f"{TRACK_NAME}_remix.wav", remix)]
if "breakcore" in globals():
    paths.append(save(f"{TRACK_NAME}_breakcore_seed{SEED}.wav", breakcore))
os.makedirs(os.path.join(OUT_DIR, "oneshots"), exist_ok=True)
for i, (t, hit) in enumerate(ONESHOTS):
    paths.append(save(f"oneshots/{TRACK_NAME}_hit{i:02d}_{t:.2f}s.wav", hit))

zip_path = os.path.join(WORK, f"{TRACK_NAME}_drums_playground.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, os.path.relpath(p, OUT_DIR))
print("\n".join(os.path.relpath(p, WORK) for p in paths))
print(f"→ {zip_path} ({os.path.getsize(zip_path)/2**20:.1f} MB)")
if IN_COLAB:
    colab_files.download(zip_path)

## 11 · 🎛️ Gradio app (+ Cloudflare tunnel)

The whole playground as a web UI, so you can upload, tweak and re-roll without touching cells:
- **Separate**: upload or record audio, or paste a YouTube or other link. You get drums and no-drums back.
- **Remix**: drum and rest levels
- **Breakcore**: every knob from section 9, plus a 🎲 button that re-rolls the seed

Players stream MP3; the download buttons give WAV. With `TUNNEL = "cloudflare"` the app is served through a free [Cloudflare quick tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/) (`*.trycloudflare.com`, no account needed). That's usually much faster for uploads and downloads than Gradio's own share link. Run sections 1–9 first, because the app reuses their functions.

> ⚠️ The tunnel URL is public while it runs: anyone with the link can use your Colab GPU. Stop this cell, or run `stop_app()`, when you're done.

In [ ]:
!pip install -q gradio

In [ ]:
import re, uuid, threading, platform
import gradio as gr

APP_DIR = os.path.join(WORK, "app_outputs")
os.makedirs(APP_DIR, exist_ok=True)


def _export(wav, stem):
    # MP3 for the in-browser player (fast over the tunnel) + WAV for download.
    base = os.path.join(APP_DIR, f"{stem}_{uuid.uuid4().hex[:6]}")
    with open(base + ".mp3", "wb") as f:
        f.write(to_mp3(wav, SR))
    sf.write(base + ".wav", np.clip(wav.numpy(), -1, 1).T, SR, subtype="PCM_16")
    return base + ".mp3", base + ".wav"


def ui_separate(path, link, start, duration, shifts, overlap, progress=gr.Progress()):
    if link:
        progress(0.05, desc="Downloading")
        path = fetch_input("youtube", link.strip())  # yt-dlp also handles plain audio URLs
    if not path:
        raise gr.Error("Upload/record some audio or paste a link first.")
    progress(0.15, desc="Decoding")
    m = load_audio(path, start, duration)
    if m.shape[-1] < SR:
        raise gr.Error("Got less than a second of audio. Check start/duration.")
    progress(0.3, desc=f"Separating on {device}")
    t0 = time.time()
    d, nd = separate(m, int(shifts), True, overlap, progress=False)
    elapsed = time.time() - t0
    name = re.sub(r"[^\w.-]+", "_", os.path.splitext(os.path.basename(path))[0])[:60]
    d_mp3, d_wav = _export(d, f"{name}_drums")
    nd_mp3, nd_wav = _export(nd, f"{name}_no_drums")
    info = (f"**{name}**: {m.shape[-1]/SR:.1f}s separated in {elapsed:.1f}s on `{device}` "
            f"({m.shape[-1]/SR/elapsed:.1f}× real-time)")
    return {"name": name, "drums": d, "no_drums": nd}, d_mp3, nd_mp3, [d_wav, nd_wav], info


def _need(state):
    if not state:
        raise gr.Error("Separate a track in the first tab first.")
    return state


def ui_remix(state, drum_db, other_db):
    st = _need(state)
    mixed = 10 ** (drum_db / 20) * st["drums"] + 10 ** (other_db / 20) * st["no_drums"]
    mixed = mixed / max(1.0, mixed.abs().max().item() / 0.99)
    mp3, wav = _export(mixed, f"{st['name']}_remix")
    return mp3, wav


def ui_breakcore(state, bpm, bars, chaos, seed, fill, bed, bed_db, kick, drive, bits, hold,
                 progress=gr.Progress()):
    st = _need(state)
    progress(0.2, desc="Chopping")
    audio, g, bsrc, info = breakcore_remix(st["drums"], st["no_drums"], int(bpm), int(bars), chaos,
                                           int(seed), int(fill), bed, bed_db, kick, drive,
                                           int(bits), int(hold))
    mp3, wav = _export(audio, f"{st['name']}_breakcore_seed{int(seed)}")
    fig = plot_pattern(g, bsrc, f"Chop pattern (seed {int(seed)}, chaos {chaos})")
    return mp3, wav, fig, info.replace("\n", "  \n")


with gr.Blocks(title="HT-Demucs Drums Playground") as demo:
    gr.Markdown("# 🥁 HT-Demucs FT drums playground\n"
                "Separate the drums, rebalance them, or turn the track into breakcore.")
    state = gr.State(None)

    with gr.Tab("1 · Separate"):
        with gr.Row():
            with gr.Column():
                in_audio = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Audio")
                in_link = gr.Textbox(label="…or a YouTube / audio URL (only audio you have rights to)",
                                     placeholder="https://")
                with gr.Row():
                    in_start = gr.Number(0, label="Start (s)")
                    in_dur = gr.Number(60, label="Duration (s, 0 = whole file)")
                with gr.Row():
                    in_shifts = gr.Slider(0, 5, 1, step=1, label="Shifts")
                    in_overlap = gr.Slider(0.0, 0.75, 0.25, step=0.05, label="Overlap")
                sep_btn = gr.Button("Separate", variant="primary")
            with gr.Column():
                sep_info = gr.Markdown()
                out_drums = gr.Audio(label="🥁 Drums", type="filepath", interactive=False)
                out_rest = gr.Audio(label="🎸 Everything except drums", type="filepath", interactive=False)
                out_files = gr.File(label="WAV downloads", file_count="multiple")
        try:
            gr.Examples([[librosa.ex("vibeace")], [librosa.ex("fishin")]], inputs=[in_audio],
                        label="Demo tracks (librosa examples, CC-licensed)")
        except Exception as e:  # offline: no examples
            print("Skipping examples:", e)

    with gr.Tab("2 · Remix"):
        with gr.Row():
            rm_drum = gr.Slider(-60, 12, 6, step=1, label="Drums (dB)")
            rm_other = gr.Slider(-60, 12, 0, step=1, label="Everything else (dB)")
        rm_btn = gr.Button("Render remix", variant="primary")
        rm_audio = gr.Audio(label="Remix", type="filepath", interactive=False)
        rm_file = gr.File(label="WAV download")

    with gr.Tab("3 · 🔥 Breakcore"):
        with gr.Row():
            with gr.Column():
                with gr.Row():
                    bc_bpm = gr.Slider(140, 260, 180, step=1, label="Target BPM")
                    bc_bars = gr.Slider(4, 64, 16, step=1, label="Bars")
                with gr.Row():
                    bc_chaos = gr.Slider(0, 1, 0.6, step=0.05, label="Chaos")
                    bc_fill = gr.Slider(0, 8, 4, step=1, label="Fill every N bars (0 = off)")
                with gr.Row():
                    bc_seed = gr.Number(7, precision=0, label="Seed")
                    dice = gr.Button("🎲 Re-roll", scale=0)
                with gr.Row():
                    bc_bed = gr.Radio(["sped-up", "original pitch", "off"], value="sped-up", label="Music bed")
                    bc_bed_db = gr.Slider(-24, 6, -4, step=1, label="Bed level (dB)")
                bc_kick = gr.Radio(["jungle", "gabber", "off"], value="jungle", label="Synth kick")
                with gr.Row():
                    bc_drive = gr.Slider(1, 10, 2.5, step=0.5, label="Drive")
                    bc_bits = gr.Slider(4, 16, 12, step=1, label="Crush bits (16 = off)")
                    bc_hold = gr.Slider(1, 8, 1, step=1, label="Sample-rate hold")
                bc_btn = gr.Button("Make breakcore", variant="primary")
            with gr.Column():
                bc_info = gr.Markdown()
                bc_audio = gr.Audio(label="Breakcore", type="filepath", interactive=False)
                bc_file = gr.File(label="WAV download")
        bc_plot = gr.Plot(label="Chop pattern")

    sep_btn.click(ui_separate, [in_audio, in_link, in_start, in_dur, in_shifts, in_overlap],
                  [state, out_drums, out_rest, out_files, sep_info])
    rm_btn.click(ui_remix, [state, rm_drum, rm_other], [rm_audio, rm_file])
    bc_inputs = [state, bc_bpm, bc_bars, bc_chaos, bc_seed, bc_fill, bc_bed, bc_bed_db,
                 bc_kick, bc_drive, bc_bits, bc_hold]
    bc_outputs = [bc_audio, bc_file, bc_plot, bc_info]
    bc_btn.click(ui_breakcore, bc_inputs, bc_outputs)
    dice.click(lambda: int(np.random.randint(0, 100_000)), None, bc_seed).then(
        ui_breakcore, bc_inputs, bc_outputs)

demo.queue(default_concurrency_limit=1)  # one GPU job at a time

In [ ]:
TUNNEL = "cloudflare"  # @param ["cloudflare", "gradio share", "local only"]
PORT   = 7860          # @param {type:"integer"}

_cf_proc = globals().get("_cf_proc")


def start_cloudflared(port, timeout=60):
    # Start a Cloudflare quick tunnel to localhost:port and return its public URL.
    global _cf_proc
    arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")
    exe = os.path.join(WORK, "cloudflared")
    if not os.path.exists(exe):
        urllib.request.urlretrieve(
            f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}", exe)
        os.chmod(exe, 0o755)
    if _cf_proc is not None and _cf_proc.poll() is None:
        _cf_proc.terminate()
    _cf_proc = subprocess.Popen([exe, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],
                                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    found, got_url, connected = {}, threading.Event(), threading.Event()

    def pump():  # keep draining cloudflared's log so it never blocks on a full pipe
        for line in _cf_proc.stderr:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m and not got_url.is_set():
                found["url"] = m.group(0); got_url.set()
            if "Registered tunnel connection" in line:
                connected.set()

    threading.Thread(target=pump, daemon=True).start()
    # cloudflared prints the URL before the tunnel is connected; wait for both.
    if not got_url.wait(timeout):
        raise RuntimeError("cloudflared didn't report a URL. Try TUNNEL = 'gradio share'.")
    if not connected.wait(timeout):
        print("⚠️  Tunnel URL issued but no edge connection registered yet. It may 530 for a bit.")
    return found["url"]


def stop_app():
    demo.close()
    if _cf_proc is not None and _cf_proc.poll() is None:
        _cf_proc.terminate()
    print("App and tunnel stopped.")


demo.close()  # free the port if this cell is re-run
if TUNNEL == "gradio share":
    demo.launch(share=True, server_port=PORT, prevent_thread_lock=True, allowed_paths=[APP_DIR])
else:
    demo.launch(server_name="127.0.0.1", server_port=PORT, prevent_thread_lock=True,
                allowed_paths=[APP_DIR], quiet=True)
    if TUNNEL == "cloudflare":
        url = start_cloudflared(PORT)
        display(Markdown(f"### 🌐 App is live at **[{url}]({url})**\n"
                         "(DNS can take ~10 s to propagate. Refresh if it doesn't load straight away.)"))
    else:
        print(f"Running on http://127.0.0.1:{PORT}")

## 12 · *(Optional)* Compare with the full 4-model ensemble

The model card says the bag's drums output **is** this sub-model's output, because the bag weights drums 1.0 for model 0 and 0 for the others. This cell checks that by running the full `htdemucs_ft` bag (about 4× the compute) and comparing. It also gives you bass, vocals and other stems to play with. With `SHIFTS > 0` the random shifts differ between runs, so expect a tiny, nonzero difference.

In [ ]:
RUN_FULL_BAG = False  # @param {type:"boolean"}

if RUN_FULL_BAG:
    t0 = time.time()
    full = run_model(bag, mix, shifts=0, split=SPLIT, overlap=OVERLAP)
    solo = run_model(model, mix, shifts=0, split=SPLIT, overlap=OVERLAP, progress=False)
    print(f"Full bag: {time.time()-t0:.1f}s (includes a shifts=0 specialist re-run)")
    diff = (full[DRUMS] - solo[DRUMS]).abs().max().item()
    print(f"max |bag drums − specialist drums| = {diff:.2e}")
    for i, s in enumerate(SOURCES):
        player(f"Full bag: {s}", full[i])
else:
    print("Set RUN_FULL_BAG = True and re-run to compare.")

---
### Notes
- **Credits:** HT-Demucs by Rouard, Massa & Défossez (Meta AI, ICASSP 2023), MIT license, [facebookresearch/demucs](https://github.com/facebookresearch/demucs). Specialist packaging by [StemSplit](https://huggingface.co/StemSplitio).
- **Memory:** with `SPLIT=True`, a full-length song fits easily on a free T4. If you hit OOM, lower `OVERLAP` or trim with `DURATION_S`.
- **Other specialists** (vocals / bass / other) are `bag.models[1..3]`. Check `bag.weights` to see which sub-model owns which stem.